# Production Digital Twin Module: Temporal Context & Alert Debouncing
This notebook demonstrates a production-grade implementation of an unsupervised anomaly detection engine on the Skoltech Anomaly Benchmark (SKAB). Unlike naive snapshot models, this pipeline engineers rolling statistical features to track sensor trends and utilizes a temporal debouncing filter to eliminate false positives caused by transient process noise.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# 1. Ingest Multivariate Sensor Streams
df = pd.read_csv('data/other/1.csv', sep=';', parse_dates=['datetime'])
features = ['Accelerometer1RMS','Accelerometer2RMS','Current','Pressure','Temperature','Thermocouple','Voltage','Volume Flow RateRMS']

In [2]:
# 2. Sliding Window Statistical Engineering
X_rolling = pd.DataFrame(index=df.index)
for col in features:
    X_rolling[f'{col}_raw'] = df[col]
    X_rolling[f'{col}_roll_mean'] = df[col].rolling(window=5, min_periods=1).mean()
    X_rolling[f'{col}_roll_std'] = df[col].rolling(window=5, min_periods=1).std().fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_rolling)

In [3]:
# 3. Unsupervised Inference Engine with Debouncing Logic
model = IsolationForest(contamination=0.08, random_state=42, n_estimators=150)
model.fit(X_scaled)
df['raw_prediction'] = (model.predict(X_scaled) == -1).astype(int)

# Apply 3-Step Industrial Alert Filter
df['twin_alert'] = df['raw_prediction'].rolling(window=3).min().fillna(0).astype(int)
print("Pipeline compiled successfully!")